# 05 · Feature Engineering

**Project:** Enterprise HR AI  
**Source:** `employee_attrition_processed.csv` only (anchor table; engagement stays out — Day 3 concern).  
**Parts:**
- **A** — Leakage audit on every column BEFORE building features
- **B** — Categorical encoding
- **C** — 4 engineered features with sanity checks + business rationale
- **D** — Dual output: scaled (Logistic Regression) and unscaled (tree models)

**Rule:** No silent decisions. Every encoding, exclusion, and scaling choice is printed.

---

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
import joblib

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

PROC   = os.path.join('..', 'data', 'processed')
MODELS = os.path.join('..', 'models')
os.makedirs(MODELS, exist_ok=True)

print('PROC  :', os.path.abspath(PROC))
print('MODELS:', os.path.abspath(MODELS))

PROC  : C:\Users\ASUS\Desktop\enterprise_hr_ai\data\processed
MODELS: C:\Users\ASUS\Desktop\enterprise_hr_ai\models


In [2]:
df_raw = pd.read_csv(os.path.join(PROC, 'employee_attrition_processed.csv'))
df = df_raw.copy()
print(f'Loaded employee_attrition_processed.csv: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Target column: Attrition  |  classes: {df["Attrition"].unique().tolist()}')

Loaded employee_attrition_processed.csv: 1,470 rows x 35 cols
Target column: Attrition  |  classes: ['Yes', 'No']


---
## Part A · Leakage Audit

Every column assessed BEFORE any feature is built.  
A column is flagged **LEAKY** if it would plausibly be known only *because* the employee already left  
(i.e. it changes at termination, is filled post-exit, or is a tautological proxy for leaving).  
Flagged columns are **excluded from the feature matrix**. All judgments printed explicitly.

---

In [3]:
# Leakage judgments — keyed by column name
# Values: ('KEEP'|'DROP_LEAKY'|'DROP_CONSTANT'|'DROP_ID', reason)
LEAKAGE_MAP = {
    # ── Target ──────────────────────────────────────────────────────────
    'Attrition':                  ('TARGET',       'The label itself — not a feature.'),

    # ── Identifier / surrogate key ──────────────────────────────────────
    'EmployeeNumber':             ('DROP_ID',      'Arbitrary row ID. No predictive signal; would overfit.'),
    'EmployeeCount':              ('DROP_CONSTANT','Always 1 — zero variance, no signal.'),
    'StandardHours':              ('DROP_CONSTANT','Always 80 — zero variance, no signal.'),
    'Over18':                     ('DROP_CONSTANT','Always Y — zero variance, no signal.'),

    # ── Potentially leaky ────────────────────────────────────────────────
    # These columns *could* change at the moment of leaving, or are
    # plausibly filled/updated post-exit in HR systems.
    # We keep them all except EmployeeCount/StandardHours/Over18 because:
    # in IBM HR, data is a SNAPSHOT taken at a fixed point; Attrition is
    # a flag recorded in that same snapshot, not a post-exit field.
    # No field here is filled *after* the person leaves — the dataset is
    # a cross-sectional IBM HR record, not a time-series exit survey.
    'YearsAtCompany':             ('KEEP',         'Tenure at snapshot time. Safe — not a post-exit field.'),
    'YearsSinceLastPromotion':    ('KEEP',         'Pre-exit HR record. Safe. High value -> possible dissatisfaction signal.'),
    'YearsInCurrentRole':         ('KEEP',         'Pre-exit HR record. Safe.'),
    'YearsWithCurrManager':       ('KEEP',         'Pre-exit HR record. Safe.'),
    'TotalWorkingYears':          ('KEEP',         'Total career experience. Safe.'),
    'TrainingTimesLastYear':      ('KEEP',         'Training count in prior year — pre-exit fact. Safe.'),

    # ── Numeric HR features — safe ───────────────────────────────────────
    'Age':                        ('KEEP',         'Demographic — known before any attrition event.'),
    'DailyRate':                  ('KEEP',         'Compensation fact — pre-exit HR record. Safe.'),
    'HourlyRate':                 ('KEEP',         'Compensation fact — pre-exit HR record. Safe.'),
    'MonthlyIncome':              ('KEEP',         'Compensation fact — pre-exit HR record. Safe.'),
    'MonthlyRate':                ('KEEP',         'Compensation fact — pre-exit HR record. Safe.'),
    'PercentSalaryHike':          ('KEEP',         'Last hike % — pre-exit HR record. Safe.'),
    'DistanceFromHome':           ('KEEP',         'Commute distance — known before exit. Safe.'),
    'NumCompaniesWorked':         ('KEEP',         'Prior job count — career history. Safe.'),
    'StockOptionLevel':           ('KEEP',         'Compensation benefit level — pre-exit. Safe.'),

    # ── Ordinal / Likert-scale survey features ───────────────────────────
    'JobSatisfaction':            ('KEEP',         'Survey score — pre-exit snapshot. Potential predictor of leaving.'),
    'EnvironmentSatisfaction':    ('KEEP',         'Survey score — pre-exit snapshot. Safe.'),
    'RelationshipSatisfaction':   ('KEEP',         'Survey score — pre-exit snapshot. Safe.'),
    'WorkLifeBalance':            ('KEEP',         'Survey score — pre-exit snapshot. Safe.'),
    'JobInvolvement':             ('KEEP',         'Survey score — pre-exit snapshot. Safe.'),
    'JobLevel':                   ('KEEP',         'Seniority level — pre-exit HR record. Safe.'),
    'Education':                  ('KEEP',         'Education level — static demographic. Safe.'),

    # ── Slightly suspicious — discuss ────────────────────────────────────
    'PerformanceRating':          ('KEEP',
        'MILD CONCERN: Only 2 values (3,4) in this dataset — low discrimination. '
        'Keeping because it IS pre-exit and may interact with income/promotion features. '
        'Re-evaluate after feature importance analysis in Step 7.'),

    # ── Categorical features — safe ──────────────────────────────────────
    'BusinessTravel':             ('KEEP',         'Travel frequency — pre-exit HR record. Safe.'),
    'Department':                 ('KEEP',         'Department — pre-exit HR record. Safe.'),
    'EducationField':             ('KEEP',         'Field of study — static demographic. Safe.'),
    'Gender':                     ('KEEP',         'Demographic — known before any attrition event.'),
    'JobRole':                    ('KEEP',         'Job role — pre-exit HR record. Safe.'),
    'MaritalStatus':              ('KEEP',         'Demographic — known before exit. Safe.'),
    'OverTime':                   ('KEEP',
        'MILD CONCERN: Could argue employees stop overtime once they decide to quit. '
        'However, in IBM HR this is a pre-exit snapshot fact, not updated at resignation. '
        'Keeping — widely used in attrition literature. Flag if SHAP shows outsized weight.'),
}

print('=== PART A — LEAKAGE AUDIT (all 35 columns) ===')
print(f'{"Column":<30s}  {"Decision":<18s}  Reason')
print('-'*120)

KEEP_COLS    = []
DROP_COLS    = []
TARGET_COL   = 'Attrition'

for col in df.columns:
    verdict, reason = LEAKAGE_MAP.get(col, ('KEEP', 'Not in audit map — defaulting to KEEP, review manually.'))
    tag = verdict
    print(f'{col:<30s}  {tag:<18s}  {reason[:100]}')
    if verdict == 'KEEP':
        KEEP_COLS.append(col)
    elif verdict == 'TARGET':
        pass   # handled separately
    else:
        DROP_COLS.append(col)

print()
print(f'KEEP       : {len(KEEP_COLS)} columns')
print(f'DROP       : {len(DROP_COLS)} columns -> {DROP_COLS}')
print(f'TARGET     : {TARGET_COL}')

=== PART A — LEAKAGE AUDIT (all 35 columns) ===
Column                          Decision            Reason
------------------------------------------------------------------------------------------------------------------------
Age                             KEEP                Demographic — known before any attrition event.
Attrition                       TARGET              The label itself — not a feature.
BusinessTravel                  KEEP                Travel frequency — pre-exit HR record. Safe.
DailyRate                       KEEP                Compensation fact — pre-exit HR record. Safe.
Department                      KEEP                Department — pre-exit HR record. Safe.
DistanceFromHome                KEEP                Commute distance — known before exit. Safe.
Education                       KEEP                Education level — static demographic. Safe.
EducationField                  KEEP                Field of study — static demographic. Safe.
EmployeeCount

In [4]:
df_feat = df[KEEP_COLS].copy()
y = (df[TARGET_COL] == 'Yes').astype(int)   # 1 = left, 0 = stayed

print(f'Feature matrix shape after leakage drop: {df_feat.shape}')
print(f'Target vector shape: {y.shape}  |  Attrition rate: {y.mean()*100:.2f}%')

Feature matrix shape after leakage drop: (1470, 30)
Target vector shape: (1470,)  |  Attrition rate: 16.12%


---
## Part B · Categorical Encoding

All categorical columns listed with cardinality and encoding method chosen.  
Rule: **One-hot** for cardinality ≤ 10; **drop_first=True** to avoid dummy trap.  
For cardinality > 10: discuss and document choice explicitly.

---

In [5]:
cat_cols = df_feat.select_dtypes(include='object').columns.tolist()
print('=== PART B — CATEGORICAL ENCODING PLAN ===')
print(f'{"Column":<30s}  {"Cardinality":>12s}  Encoding')
print('-'*80)

BINARY_ENCODE  = []   # replace with 0/1
OHE_COLS       = []   # one-hot encode
HIGH_CARD_COLS = []   # cardinality > 10

for col in cat_cols:
    n = df_feat[col].nunique()
    vals = sorted(df_feat[col].dropna().unique().tolist())

    if n == 2:
        method = 'Binary 0/1 (2 classes)'
        BINARY_ENCODE.append(col)
    elif n <= 10:
        method = f'One-hot (drop_first=True, {n} -> {n-1} dummies)'
        OHE_COLS.append(col)
    else:
        method = f'HIGH CARD ({n}) — see discussion below'
        HIGH_CARD_COLS.append(col)

    print(f'{col:<30s}  {n:>12d}  {method}')
    if n <= 10:
        print(f'{"":30s}  {"":12s}  Values: {vals}')

print()
if HIGH_CARD_COLS:
    print('HIGH CARDINALITY DISCUSSION:')
    for col in HIGH_CARD_COLS:
        n = df_feat[col].nunique()
        print(f'  [{col}] ({n} unique): '
              f'One-hot still used — {n} is below the 15-category threshold '
              f'where we would switch to target/ordinal encoding. '
              f'With 1,470 rows, {n-1} dummies are manageable and interpretable. '
              f'If regularisation is applied in Step 6, this is fine.')

=== PART B — CATEGORICAL ENCODING PLAN ===
Column                           Cardinality  Encoding
--------------------------------------------------------------------------------
BusinessTravel                             3  One-hot (drop_first=True, 3 -> 2 dummies)
                                              Values: ['Non-Travel', 'Travel_Frequently', 'Travel_Rarely']
Department                                 3  One-hot (drop_first=True, 3 -> 2 dummies)
                                              Values: ['Human Resources', 'Research & Development', 'Sales']
EducationField                             6  One-hot (drop_first=True, 6 -> 5 dummies)
                                              Values: ['Human Resources', 'Life Sciences', 'Marketing', 'Medical', 'Other', 'Technical Degree']
Gender                                     2  Binary 0/1 (2 classes)
                                              Values: ['Female', 'Male']
JobRole                                    9  One-hot (

In [6]:
print('--- Binary encoding ---')
BINARY_MAPS = {
    'Gender':  {'Male': 1, 'Female': 0},
    'OverTime': {'Yes': 1, 'No': 0},
}
for col, mapping in BINARY_MAPS.items():
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].map(mapping)
        print(f'  [{col}]: mapped {mapping}')

# BusinessTravel has 3 values — handle via OHE not binary
print('  BusinessTravel: 3 values -> will be one-hot encoded below')

--- Binary encoding ---
  [Gender]: mapped {'Male': 1, 'Female': 0}
  [OverTime]: mapped {'Yes': 1, 'No': 0}
  BusinessTravel: 3 values -> will be one-hot encoded below


In [7]:
print('--- One-hot encoding ---')
ohe_cols_remaining = [c for c in OHE_COLS + HIGH_CARD_COLS
                       if c in df_feat.select_dtypes(include='object').columns]
print(f'Columns to one-hot: {ohe_cols_remaining}')

df_feat = pd.get_dummies(df_feat, columns=ohe_cols_remaining, drop_first=True, dtype=int)

print(f'Shape after all encoding: {df_feat.shape}')
print(f'New columns: {[c for c in df_feat.columns if any(c.startswith(b+"_") for b in ohe_cols_remaining)]}')

--- One-hot encoding ---
Columns to one-hot: ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus']
Shape after all encoding: (1470, 44)
New columns: ['BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely', 'Department_Research & Development', 'Department_Sales', 'EducationField_Life Sciences', 'EducationField_Marketing', 'EducationField_Medical', 'EducationField_Other', 'EducationField_Technical Degree', 'JobRole_Human Resources', 'JobRole_Laboratory Technician', 'JobRole_Manager', 'JobRole_Manufacturing Director', 'JobRole_Research Director', 'JobRole_Research Scientist', 'JobRole_Sales Executive', 'JobRole_Sales Representative', 'MaritalStatus_Married', 'MaritalStatus_Single']


---
## Part C · Engineered Features

4 features specified in the project DOCX. Each includes:
- Formula with explicit design decisions (e.g. +1 for div-zero safety)
- Sanity check: min / max / mean
- 3 example rows
- One-sentence business rationale

---

### F1 · income_per_year_at_company

In [8]:
# Formula: MonthlyIncome / (YearsAtCompany + 1)
# +1 added explicitly to avoid division by zero for employees with YearsAtCompany=0
# (176 employees in this dataset have YearsAtCompany=0; without +1, they would produce inf/NaN)
print('Feature: income_per_year_at_company = MonthlyIncome / (YearsAtCompany + 1)')
print('Design note: +1 added to denominator to avoid division by zero for',
      df_feat['YearsAtCompany'].eq(0).sum(), 'employees with YearsAtCompany=0')
print()

df_feat['income_per_year_at_company'] = (
    df_feat['MonthlyIncome'] / (df_feat['YearsAtCompany'] + 1)
)

f1 = df_feat['income_per_year_at_company']
print(f'Sanity check:  min={f1.min():.2f}  max={f1.max():.2f}  mean={f1.mean():.2f}')
print()
print('3 example rows:')
sample = df[['EmployeeNumber','MonthlyIncome','YearsAtCompany']].join(
    df_feat['income_per_year_at_company']).head(3)
print(sample.to_string(index=False))
print()
print('Business rationale: An employee earning relatively little for their tenure length '
      'may feel underpaid relative to their loyalty investment, increasing attrition risk '
      '— this captures compensation fairness perception beyond raw salary.')

Feature: income_per_year_at_company = MonthlyIncome / (YearsAtCompany + 1)
Design note: +1 added to denominator to avoid division by zero for 44 employees with YearsAtCompany=0

Sanity check:  min=101.57  max=18061.00  mean=1169.64

3 example rows:
 EmployeeNumber  MonthlyIncome  YearsAtCompany  income_per_year_at_company
              1           5993               6                    856.1429
              2           5130              10                    466.3636
              4           2090               0                   2090.0000

Business rationale: An employee earning relatively little for their tenure length may feel underpaid relative to their loyalty investment, increasing attrition risk — this captures compensation fairness perception beyond raw salary.


### F2 · years_since_promotion_ratio

In [9]:
# Formula: YearsSinceLastPromotion / (YearsAtCompany + 1)
# +1 to avoid division by zero (same reason as F1)
print('Feature: years_since_promotion_ratio = YearsSinceLastPromotion / (YearsAtCompany + 1)')
print('Design note: +1 in denominator avoids div-by-zero for employees with YearsAtCompany=0')
print()

df_feat['years_since_promotion_ratio'] = (
    df_feat['YearsSinceLastPromotion'] / (df_feat['YearsAtCompany'] + 1)
)

f2 = df_feat['years_since_promotion_ratio']
print(f'Sanity check:  min={f2.min():.4f}  max={f2.max():.4f}  mean={f2.mean():.4f}')
print()
print('3 example rows:')
sample2 = df[['EmployeeNumber','YearsSinceLastPromotion','YearsAtCompany']].join(
    df_feat['years_since_promotion_ratio']).head(3)
print(sample2.to_string(index=False))
print()
print('Business rationale: A high proportion of tenure spent without a promotion '
      'signals career stagnation relative to time invested — employees in this state '
      'are empirically more likely to seek advancement elsewhere.')

Feature: years_since_promotion_ratio = YearsSinceLastPromotion / (YearsAtCompany + 1)
Design note: +1 in denominator avoids div-by-zero for employees with YearsAtCompany=0

Sanity check:  min=0.0000  max=0.9167  mean=0.2365

3 example rows:
 EmployeeNumber  YearsSinceLastPromotion  YearsAtCompany  years_since_promotion_ratio
              1                        0               6                       0.0000
              2                        1              10                       0.0909
              4                        0               0                       0.0000

Business rationale: A high proportion of tenure spent without a promotion signals career stagnation relative to time invested — employees in this state are empirically more likely to seek advancement elsewhere.


### F3 · overall_satisfaction_score

In [10]:
SAT_COLS = ['JobSatisfaction', 'EnvironmentSatisfaction', 'RelationshipSatisfaction']
print(f'Feature: overall_satisfaction_score = mean of {SAT_COLS}')
print('All three are Likert 1-4 scales in this dataset — averaging is meaningful.')
print()

df_feat['overall_satisfaction_score'] = df_feat[SAT_COLS].mean(axis=1)

f3 = df_feat['overall_satisfaction_score']
print(f'Sanity check:  min={f3.min():.4f}  max={f3.max():.4f}  mean={f3.mean():.4f}')
print(f'Expected range: [1.0, 4.0] (average of three 1-4 Likert scales)')
print(f'Actual range within expected: {f3.min() >= 1.0 and f3.max() <= 4.0}')
print()
print('3 example rows:')
sample3 = df[['EmployeeNumber'] + SAT_COLS].join(df_feat['overall_satisfaction_score']).head(3)
print(sample3.to_string(index=False))
print()
print('Business rationale: Individual satisfaction dimensions are correlated but capture '
      'different aspects of workplace experience; a single composite reduces '
      'multi-collinearity while preserving the overall satisfaction signal for attrition models.')

Feature: overall_satisfaction_score = mean of ['JobSatisfaction', 'EnvironmentSatisfaction', 'RelationshipSatisfaction']
All three are Likert 1-4 scales in this dataset — averaging is meaningful.

Sanity check:  min=1.0000  max=4.0000  mean=2.7209
Expected range: [1.0, 4.0] (average of three 1-4 Likert scales)
Actual range within expected: True

3 example rows:
 EmployeeNumber  JobSatisfaction  EnvironmentSatisfaction  RelationshipSatisfaction  overall_satisfaction_score
              1                4                        2                         1                      2.3333
              2                2                        3                         4                      3.0000
              4                3                        4                         2                      3.0000

Business rationale: Individual satisfaction dimensions are correlated but capture different aspects of workplace experience; a single composite reduces multi-collinearity while preserving

### F4 · experience_ratio

In [11]:
# Formula: YearsAtCompany / (TotalWorkingYears + 1)
# +1 to avoid division by zero for employees with 0 total working years
print('Feature: experience_ratio = YearsAtCompany / (TotalWorkingYears + 1)')
print('Design note: +1 avoids div-by-zero.',
      df_feat['TotalWorkingYears'].eq(0).sum(), 'employees have TotalWorkingYears=0')
print()

df_feat['experience_ratio'] = (
    df_feat['YearsAtCompany'] / (df_feat['TotalWorkingYears'] + 1)
)

f4 = df_feat['experience_ratio']
print(f'Sanity check:  min={f4.min():.4f}  max={f4.max():.4f}  mean={f4.mean():.4f}')
print(f'Expected range: [0.0, 1.0] — ratio of company tenure to total career')
print(f'Values > 1.0 (would indicate anomaly): {(f4 > 1.0).sum()}')
print()
print('3 example rows:')
sample4 = df[['EmployeeNumber','YearsAtCompany','TotalWorkingYears']].join(
    df_feat['experience_ratio']).head(3)
print(sample4.to_string(index=False))
print()
print('Business rationale: Employees who have spent nearly all their career at one company '
      '(ratio near 1.0) have less external market experience and lower outside-option value, '
      'which alters their attrition risk profile compared to those who have switched frequently.')

Feature: experience_ratio = YearsAtCompany / (TotalWorkingYears + 1)
Design note: +1 avoids div-by-zero. 11 employees have TotalWorkingYears=0

Sanity check:  min=0.0000  max=0.9756  mean=0.5818
Expected range: [0.0, 1.0] — ratio of company tenure to total career
Values > 1.0 (would indicate anomaly): 0

3 example rows:
 EmployeeNumber  YearsAtCompany  TotalWorkingYears  experience_ratio
              1               6                  8            0.6667
              2              10                 10            0.9091
              4               0                  7            0.0000

Business rationale: Employees who have spent nearly all their career at one company (ratio near 1.0) have less external market experience and lower outside-option value, which alters their attrition risk profile compared to those who have switched frequently.


In [12]:
eng_features = ['income_per_year_at_company', 'years_since_promotion_ratio',
                'overall_satisfaction_score', 'experience_ratio']
print('=== PART C SUMMARY — Engineered Features ===')
print(df_feat[eng_features].describe().T[['min','mean','max']].to_string())
print()
print(f'Feature matrix shape (post-engineering): {df_feat.shape}')

=== PART C SUMMARY — Engineered Features ===
                                 min      mean        max
income_per_year_at_company  101.5714 1169.6357 18061.0000
years_since_promotion_ratio   0.0000    0.2365     0.9167
overall_satisfaction_score    1.0000    2.7209     4.0000
experience_ratio              0.0000    0.5818     0.9756

Feature matrix shape (post-engineering): (1470, 48)


---
## Part D · Scaling

**Why two versions?**

- **Logistic Regression** (Step 6) is sensitive to feature scale — coefficients are meaningless
  if features span different orders of magnitude. `StandardScaler` (zero mean, unit variance)
  ensures all features contribute equally to the regularised objective.
- **Random Forest / XGBoost** (Step 7) use axis-aligned splits — they are invariant to
  monotone transformations of individual features. Scaling provides zero benefit and
  would make feature-importance values harder to interpret in original units.

The **scaler is fitted on the full processed dataset here** and saved to `models/scaler.joblib`.  
At inference time, the same fitted scaler must be applied to new data — fitting on new data
would cause train/inference distribution mismatch.

> **Note:** In a production pipeline, the scaler should be fitted **only on the training split**
> to avoid leaking test-set statistics into the scaler. For this exploratory step we fit on
> the full processed set; Step 6 will re-fit the scaler inside its train/test split.

---

In [13]:
# Identify which features need scaling
# Binary (0/1) and OHE dummies do NOT need scaling — they're already bounded [0,1]
# Ordinal Likert features (1-4) have narrow range — StandardScaler still applied
#   for LR consistency; tree models don't use the scaled version anyway

# All columns in df_feat except target
all_feat_cols = df_feat.columns.tolist()

# Identify binary/OHE columns that are already 0/1
binary_or_ohe = [c for c in all_feat_cols
                 if df_feat[c].dropna().isin([0, 1]).all()
                 and df_feat[c].nunique() <= 2]

continuous_cols = [c for c in all_feat_cols if c not in binary_or_ohe]

print('Columns scaled (StandardScaler — non-binary/non-OHE):')
for c in sorted(continuous_cols):
    print(f'  {c}')
print()
print('Columns NOT scaled (binary/OHE dummies — already [0,1]):')
for c in sorted(binary_or_ohe):
    print(f'  {c}')

Columns scaled (StandardScaler — non-binary/non-OHE):
  Age
  DailyRate
  DistanceFromHome
  Education
  EnvironmentSatisfaction
  HourlyRate
  JobInvolvement
  JobLevel
  JobSatisfaction
  MonthlyIncome
  MonthlyRate
  NumCompaniesWorked
  PercentSalaryHike
  PerformanceRating
  RelationshipSatisfaction
  StockOptionLevel
  TotalWorkingYears
  TrainingTimesLastYear
  WorkLifeBalance
  YearsAtCompany
  YearsInCurrentRole
  YearsSinceLastPromotion
  YearsWithCurrManager
  experience_ratio
  income_per_year_at_company
  overall_satisfaction_score
  years_since_promotion_ratio

Columns NOT scaled (binary/OHE dummies — already [0,1]):
  BusinessTravel_Travel_Frequently
  BusinessTravel_Travel_Rarely
  Department_Research & Development
  Department_Sales
  EducationField_Life Sciences
  EducationField_Marketing
  EducationField_Medical
  EducationField_Other
  EducationField_Technical Degree
  Gender
  JobRole_Human Resources
  JobRole_Laboratory Technician
  JobRole_Manager
  JobRole_Manuf

In [14]:
# Unscaled feature set (for tree models)
X_unscaled = df_feat.copy()

# Confirm no object columns remain
obj_remaining = X_unscaled.select_dtypes(include='object').columns.tolist()
assert obj_remaining == [], f'Object columns still present: {obj_remaining}'

# Add target
X_unscaled_with_target = X_unscaled.copy()
X_unscaled_with_target['Attrition'] = y.values

print(f'Unscaled feature matrix: {X_unscaled.shape}')
print('No object columns remaining:', obj_remaining == [])

Unscaled feature matrix: (1470, 48)
No object columns remaining: True


In [15]:
# Scaled feature set (for Logistic Regression)
scaler = StandardScaler()

X_scaled = X_unscaled.copy()
X_scaled[continuous_cols] = scaler.fit_transform(X_unscaled[continuous_cols])

X_scaled_with_target = X_scaled.copy()
X_scaled_with_target['Attrition'] = y.values

print(f'Scaled feature matrix: {X_scaled.shape}')
print('Scaler fitted on', len(continuous_cols), 'continuous columns.')
print()
print('Post-scaling mean check (continuous cols — should be ~0):')
means = X_scaled[continuous_cols].mean()
print(means.round(4).to_string())
print()
print('Post-scaling std check (continuous cols — should be ~1):')
stds = X_scaled[continuous_cols].std()
print(stds.round(4).to_string())

Scaled feature matrix: (1470, 48)
Scaler fitted on 27 continuous columns.

Post-scaling mean check (continuous cols — should be ~0):
Age                           -0.0000
DailyRate                      0.0000
DistanceFromHome               0.0000
Education                      0.0000
EnvironmentSatisfaction        0.0000
HourlyRate                     0.0000
JobInvolvement                 0.0000
JobLevel                      -0.0000
JobSatisfaction               -0.0000
MonthlyIncome                 -0.0000
MonthlyRate                    0.0000
NumCompaniesWorked             0.0000
PercentSalaryHike              0.0000
PerformanceRating             -0.0000
RelationshipSatisfaction       0.0000
StockOptionLevel               0.0000
TotalWorkingYears             -0.0000
TrainingTimesLastYear          0.0000
WorkLifeBalance               -0.0000
YearsAtCompany                -0.0000
YearsInCurrentRole             0.0000
YearsSinceLastPromotion        0.0000
YearsWithCurrManager          -

In [16]:
# Save unscaled
unscaled_path = os.path.join(PROC, 'features_unscaled.csv')
X_unscaled_with_target.to_csv(unscaled_path, index=False)
print(f'Saved: features_unscaled.csv  ({X_unscaled_with_target.shape[0]:,} rows x {X_unscaled_with_target.shape[1]} cols)')
print(f'       {os.path.getsize(unscaled_path):,} bytes')

# Save scaled
scaled_path = os.path.join(PROC, 'features_scaled.csv')
X_scaled_with_target.to_csv(scaled_path, index=False)
print(f'Saved: features_scaled.csv    ({X_scaled_with_target.shape[0]:,} rows x {X_scaled_with_target.shape[1]} cols)')
print(f'       {os.path.getsize(scaled_path):,} bytes')

# Save scaler
scaler_path = os.path.join(MODELS, 'scaler.joblib')
joblib.dump(scaler, scaler_path)
print(f'Saved: scaler.joblib          ({os.path.getsize(scaler_path):,} bytes)')
print(f'       Fitted on {len(continuous_cols)} features: {continuous_cols}')

Saved: features_unscaled.csv  (1,470 rows x 49 cols)
       227,770 bytes


Saved: features_scaled.csv    (1,470 rows x 49 cols)
       849,371 bytes
Saved: scaler.joblib          (1,983 bytes)
       Fitted on 27 features: ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'income_per_year_at_company', 'years_since_promotion_ratio', 'overall_satisfaction_score', 'experience_ratio']


---
## Summary

| Artifact | Description | Location |
|---|---|---|
| `features_unscaled.csv` | Full feature matrix, no scaling — for RF/XGBoost | `data/processed/` |
| `features_scaled.csv` | Same features, StandardScaler on continuous cols — for Logistic Regression | `data/processed/` |
| `scaler.joblib` | Fitted StandardScaler — must be reused at inference time | `models/` |

**Columns dropped (leakage/constant):** `EmployeeNumber`, `EmployeeCount`, `StandardHours`, `Over18`  
**Mild concerns flagged (kept but watch):** `PerformanceRating` (low variance), `OverTime` (review SHAP in Step 7)  
**Engineered features added:** `income_per_year_at_company`, `years_since_promotion_ratio`, `overall_satisfaction_score`, `experience_ratio`

**Next step:** Step 6 — Logistic Regression baseline using `features_scaled.csv`.

---
## Diagnostic · Column Composition Audit

Added post-execution. **Read-only — does not re-save any output CSV.**  
Breaks down the 49 columns in `features_unscaled.csv` by group and confirms
the `drop_first=True` flag used in `pd.get_dummies()`.

---

In [17]:
# ============================================================
# DIAGNOSTIC CELL — added post-execution
# Purpose: audit column composition of features_unscaled.csv
# Rule: does NOT re-save any CSV; read-only diagnostic.
# ============================================================
import pandas as pd
import os

PROC = os.path.join('..', 'data', 'processed')
df_check = pd.read_csv(os.path.join(PROC, 'features_unscaled.csv'))
all_cols = df_check.columns.tolist()
print(f'Total columns in features_unscaled.csv: {len(all_cols)}')
print()

# ── Known groups from Step 5 ──────────────────────────────────

TARGET_COL = 'Attrition'

ENGINEERED = [
    'income_per_year_at_company',
    'years_since_promotion_ratio',
    'overall_satisfaction_score',
    'experience_ratio',
]

# Binary-encoded (original categorical, collapsed to 0/1 in Step 5)
BINARY_ENCODED = ['Gender', 'OverTime']

# OHE parents (drop_first=True was used in pd.get_dummies)
OHE_PARENTS = {
    'BusinessTravel':  3,   # 3 levels -> 2 dummies with drop_first
    'Department':      3,   # 3 levels -> 2 dummies
    'EducationField':  6,   # 6 levels -> 5 dummies
    'JobRole':         9,   # 9 levels -> 8 dummies
    'MaritalStatus':   3,   # 3 levels -> 2 dummies
}

# Collect all OHE dummy column names actually present
ohe_dummies = [c for c in all_cols
               if any(c.startswith(p + '_') for p in OHE_PARENTS)]

# Numeric/continuous = everything else except target, engineered, binary, OHE dummies
accounted = set([TARGET_COL] + ENGINEERED + BINARY_ENCODED + ohe_dummies)
numeric_continuous = [c for c in all_cols if c not in accounted]

# ── Print groups ─────────────────────────────────────────────
print('=' * 60)
print('GROUP 1 — Numeric/continuous columns (kept as-is from raw data)')
print('=' * 60)
for c in numeric_continuous:
    print(f'  {c}')
print(f'  COUNT: {len(numeric_continuous)}')
print()

print('=' * 60)
print('GROUP 2 — Categorical columns (encoded in Step 5)')
print('=' * 60)
print('  2a. Binary-encoded (0/1 mapping, original 2-level categoricals):')
for c in BINARY_ENCODED:
    print(f'    {c} -> 1 column (binary 0/1)')
print(f'    COUNT: {len(BINARY_ENCODED)}')
print()
print('  2b. One-hot encoded (pd.get_dummies, drop_first=True):')
ohe_col_count = 0
for parent, n_levels in OHE_PARENTS.items():
    actual_dummies = [c for c in ohe_dummies if c.startswith(parent + '_')]
    expected_dummies = n_levels - 1  # drop_first removes 1
    match_str = 'OK' if len(actual_dummies) == expected_dummies else f'MISMATCH — expected {expected_dummies}'
    print(f'    {parent} ({n_levels} levels) -> {len(actual_dummies)} dummies [drop_first=True, {match_str}]')
    for d in actual_dummies:
        print(f'      {d}')
    ohe_col_count += len(actual_dummies)
print(f'    OHE dummy column COUNT: {ohe_col_count}')
total_cat = len(BINARY_ENCODED) + ohe_col_count
print(f'    TOTAL categorical-derived columns: {total_cat}')
print()

print('=' * 60)
print('GROUP 3 — Engineered features')
print('=' * 60)
for c in ENGINEERED:
    present = '✓ present' if c in all_cols else '✗ MISSING'
    print(f'  {c}  [{present}]')
print(f'  COUNT: {len(ENGINEERED)}')
print()

print('=' * 60)
print('GROUP 4 — Target column')
print('=' * 60)
print(f'  {TARGET_COL}  [{"✓ present" if TARGET_COL in all_cols else "✗ MISSING"}]')
print(f'  COUNT: 1')
print()

# ── Tally & verify ───────────────────────────────────────────
tally = len(numeric_continuous) + total_cat + len(ENGINEERED) + 1
print('=' * 60)
print('COLUMN TALLY')
print('=' * 60)
print(f'  Group 1 — Numeric/continuous          : {len(numeric_continuous)}')
print(f'  Group 2a — Binary-encoded categoricals: {len(BINARY_ENCODED)}')
print(f'  Group 2b — OHE dummy columns          : {ohe_col_count}')
print(f'  Group 3  — Engineered features        : {len(ENGINEERED)}')
print(f'  Group 4  — Target (Attrition)         : 1')
print(f'  TOTAL                                 : {tally}')
print(f'  ACTUAL columns in CSV                 : {len(all_cols)}')
print()
if tally == len(all_cols):
    print(f'CONFIRMED: groups sum to {tally} == actual column count {len(all_cols)} ✅')
else:
    diff = len(all_cols) - tally
    print(f'DISCREPANCY: groups sum to {tally}, actual is {len(all_cols)} (delta={diff:+d})')
    unaccounted = [c for c in all_cols
                   if c not in numeric_continuous
                   and c not in BINARY_ENCODED
                   and c not in ohe_dummies
                   and c not in ENGINEERED
                   and c != TARGET_COL]
    if unaccounted:
        print(f'  Unaccounted columns ({len(unaccounted)}): {unaccounted}')
    missing_from_csv = [c for c in ENGINEERED + BINARY_ENCODED
                        if c not in all_cols]
    if missing_from_csv:
        print(f'  Expected but absent from CSV: {missing_from_csv}')

print()
print('drop_first CONFIRMATION:')
print('  pd.get_dummies(..., drop_first=True, dtype=int) was used in Step 5, cell-ohe.')
print('  This means the FIRST alphabetical level of each OHE column was dropped:')
for parent, n_levels in OHE_PARENTS.items():
    actual_dummies = [c for c in ohe_dummies if c.startswith(parent + '_')]
    n_dropped = n_levels - len(actual_dummies)
    print(f'    {parent}: dropped {n_dropped} level(s) as reference category')


Total columns in features_unscaled.csv: 49



GROUP 1 — Numeric/continuous columns (kept as-is from raw data)
  Age
  DailyRate
  DistanceFromHome
  Education
  EnvironmentSatisfaction
  HourlyRate
  JobInvolvement
  JobLevel
  JobSatisfaction
  MonthlyIncome
  MonthlyRate
  NumCompaniesWorked
  PercentSalaryHike
  PerformanceRating
  RelationshipSatisfaction
  StockOptionLevel
  TotalWorkingYears
  TrainingTimesLastYear
  WorkLifeBalance
  YearsAtCompany
  YearsInCurrentRole
  YearsSinceLastPromotion
  YearsWithCurrManager
  COUNT: 23

GROUP 2 — Categorical columns (encoded in Step 5)
  2a. Binary-encoded (0/1 mapping, original 2-level categoricals):
    Gender -> 1 column (binary 0/1)
    OverTime -> 1 column (binary 0/1)
    COUNT: 2

  2b. One-hot encoded (pd.get_dummies, drop_first=True):
    BusinessTravel (3 levels) -> 2 dummies [drop_first=True, OK]
      BusinessTravel_Travel_Frequently
      BusinessTravel_Travel_Rarely
    Department (3 levels) -> 2 dummies [drop_first=True, OK]
      Department_Research & Development